In [8]:
import numpy as np
import pandas as pd
from sympy import limit
import wandb

api = wandb.Api(
    api_key="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",
    timeout=60
)

# 1. Get last 10 runs (sorted by creation time descending)
runs = api.runs(
    "eibl-usc/graph-clip",
    filters={
        # "display_name": {"$regex": "trained_on_.._eval_on_.._..?_shot_.+"},
        # "config.dataset": "ukr_rus_twitter",
        # "config.task_name": "neighbor_matching",
    },
    order="-created_at",
    per_page=10,
    # limit to 10:
    lazy=False
)


In [9]:

rows = []
for run in runs:
    attrs = getattr(run, "_attrs", {}) or {}
    params = ((attrs.get("config") or {}).get("params") or {})
    summary = attrs.get("summaryMetrics") or {}

    rows.append({
        "run_id": attrs.get("name"),
        "display_name": attrs.get("displayName"),
        "state": attrs.get("state"),
        "dataset": params.get("dataset"),
        "task_name": params.get("task_name"),
        "prefix": params.get("prefix"),
        "pretrained_model_run": params.get("pretrained_model_run"),
        "n_shots": params.get("n_shots"),
        "n_way": params.get("n_way"),
        "n_query": params.get("n_query"),
        "zero_shot": params.get("zero_shot"),
        "test_accuracy": summary.get("test_accuracy"),
        "train_accuracy": summary.get("train_accuracy"),
        "test_f1": summary.get("test_f1"),
        "test_roc_auc": summary.get("test_roc_auc"),
        "created_at": attrs.get("createdAt"),
        'steps': attrs['historyKeys']['keys'].get('_step', {}).get('typeCounts', [{}])[0].get('count', np.nan),
        'tags': attrs.get("tags", []),
        'input_dim': params.get("input_dim"),
        'feature_subset': params.get('feature_subset', params.get('midterm_feature_subset')),
        'eval_only': params.get("eval_only"),
        'graph': params.get('root', 'nan') + '/' + params.get('graph_filename', 'nan')
    })
df = pd.DataFrame(rows)
df["task_name"] = df["task_name"].map({
    "neighbor_matching": "nm",
    "temporal_link_prediction": "lp",
    "classification": "pl",
})
df['created_at'] = pd.to_datetime(df['created_at'])
df["shot_label"] = df.apply(lambda r: 0 if bool(r.get("zero_shot", False)) else r["n_shots"], axis=1)
# df['is_eval'] = df['display_name'].str.contains(r"eval")
# plot_df = df[df["eval1_task"].isin(EVAL_TASKS) & df["train1_task"].eq("nm")].copy()
df['eval1_dataset'] = df['dataset']
df['trained_on_display_name'] = df.pretrained_model_run.str.findall("(.+)/(.+)/((checkpoint/(.+)\.ckpt)|state_dict)$").str[0].str[1]
df['month/day'] = df['created_at'].dt.month.astype(str) + '/' + df['created_at'].dt.day.astype(str)
df = df.sort_values('created_at', ascending=False)
mask = df['trained_on_display_name'].isin(df['display_name'])
existing_trained_on_display_names = df.trained_on_display_name[mask]
df__ = df.copy()

df['shots'] = pd.to_numeric(df.n_shots.fillna(-1)).astype(int)

df = df[df.dataset.isin(['midterm', 'ukr_rus_twitter', 'covid19_twitter', 'covid_political', 'ukr_rus_suspended', 'election2020'])]

df["train_tuples"] = None  # or use pd.Series(dtype=object)
df["run_list"] = None  # or use pd.Series(dtype=object)

for i, row in list(df.iterrows()):
    train_list = []
    run_list = []
    row_ = row.copy()
    # print(row_.display_name)

    while pd.notna(row_.trained_on_display_name):
        # print("->", row_.trained_on_display_name)
        matches = df[df.display_name.eq(row_.trained_on_display_name)]
        if len(matches) != 1:
            print('warn:', row_.trained_on_display_name, )
            break
        row_ = matches.iloc[0]
        train_list.append((row_["dataset"], row_["task_name"], row_.get('shots', -1)))
        run_list.append(row_.run_id)

    # print('\t', train_list)
    df.at[i, "train_tuples"] = train_list
    df.at[i, "run_list"] = run_list


# x = df.train_idx.explode().dropna().groupby(level=0).agg(list)
# df['train_idx'] = np.nan
# df['train_idx'] = x
df = df[~df.display_name.str.contains('train\d*_')]

df['train_id'] = df.train_tuples.apply(lambda x: [f'{d}+{t}({s})' for (d,t,s) in x]).apply('>'.join).replace({'':np.nan})
df['eval_only'] = df['eval_only'].fillna(False)
df.loc[df.eval_only, 'eval_dataset'] = df['dataset']
df.loc[df.eval_only, 'eval_task'] = df['task_name']
df.loc[df.eval_only, 'eval_id'] = df.eval_dataset + "+" + df.eval_task + "(" + df.shots.astype(str) + ")"
df.loc[df.eval_only, 'run_idx'] = df.run_list.apply('>'.join).replace('', 'nan') + "|" + df.eval_id
df['seq'] = df.train_id + "|" + df.eval_id

df['auc'] = df.test_roc_auc
df['f1'] = df.test_f1

warn: exp3_train2_covid_nm_to_ukr_rus_nm_16_04_2026_13_39_26
warn: exp15_train2_ukr_rus_nm_to_covid_nm_23_04_2026_13_33_55
warn: train2_midterm_nm_to_covid_nm_16_04_2026_10_07_00
warn: exp13_train2_covid_nm_to_midterm_nm_23_04_2026_13_24_00
warn: exp3_train2_covid_nm_to_ukr_rus_nm_16_04_2026_13_39_26
warn: exp13_train2_covid_nm_to_midterm_nm_23_04_2026_13_24_00
warn: exp2_train2_midterm_nm_to_ukr_rus_nm_16_04_2026_10_32_38
warn: exp14_train2_ukr_rus_nm_to_midterm_nm_23_04_2026_13_26_26
warn: train2_midterm_nm_to_covid_nm_16_04_2026_10_07_00
warn: exp3_train2_covid_nm_to_ukr_rus_nm_16_04_2026_13_39_26
warn: exp3_train2_covid_nm_to_ukr_rus_nm_16_04_2026_13_39_26
warn: train2_midterm_nm_to_covid_nm_16_04_2026_10_07_00
warn: exp14_train2_ukr_rus_nm_to_midterm_nm_23_04_2026_13_26_26
warn: exp15_train2_ukr_rus_nm_to_covid_nm_23_04_2026_13_33_55
warn: exp14_train2_ukr_rus_nm_to_midterm_nm_23_04_2026_13_26_26
warn: exp15_train2_ukr_rus_nm_to_covid_nm_23_04_2026_13_33_55
warn: exp14_train2_ukr_

In [10]:
graphs = ["/scratch1/eibl/data/covid_political/graphs/retweet_graph.pt",
"/scratch1/eibl/data/election2020/graphs/retweet_graph.pt",
"/scratch1/eibl/data/ukr_rus_suspended/graphs/retweet_graph.pt",
"/scratch1/eibl/data/covid19_twitter/graphs/retweet_graph_1p5m_hf03_labeled.pt",
"/scratch1/eibl/data/ukr_rus_twitter/graphs/retweet_graph_1p5m_hf03_political_labels.pt",
"/scratch1/eibl/data/midterm/graphs/retweet_graph_1p5m.pt",]

df = df[df.graph.isin(graphs)]

df = df.dropna(subset=['auc'])

In [14]:
ts = df[df.tags.astype(str).str.contains('remaining_train2_eval_04_27')]['created_at']
start, end = ts.min(), ts.max()   # safer than assuming order
remaining_train2_eval_04_27 = df[df['created_at'].between(start, end)]
remaining_train2_eval_04_27 = remaining_train2_eval_04_27[~remaining_train2_eval_04_27.run_id.eq("u3pd8020")]
remaining_train2_eval_04_27.to_csv("train2/remaining_train2_eval_04_27.csv", index=False)
remaining_train2_eval_04_27

,run_id,display_name,state,dataset,task_name,prefix,pretrained_model_run,n_shots,n_way,n_query,...,train_tuples,run_list,train_id,eval_dataset,eval_task,eval_id,run_idx,seq,auc,f1
68,0qntwfd1,eval_exp9_covid_nm_to_ukr_rus_lp_to_election20...,finished,election2020,pl,eval_exp9_covid_nm_to_ukr_rus_lp_to_election20...,/scratch1/singhama/data/experiments/exp9_train...,3.0,2.0,12.0,...,[],[],NaN,election2020,pl,election2020+pl(3),nan|election2020+pl(3),NaN,0.587430,0.577726
69,ljjfeuwu,eval_exp9_covid_nm_to_ukr_rus_lp_to_election20...,finished,election2020,nm,eval_exp9_covid_nm_to_ukr_rus_lp_to_election20...,/scratch1/singhama/data/experiments/exp9_train...,3.0,3.0,12.0,...,[],[],NaN,election2020,nm,election2020+nm(3),nan|election2020+nm(3),NaN,0.632019,0.448538
70,1j3usyar,eval_exp9_covid_nm_to_ukr_rus_lp_to_election20...,finished,election2020,pl,eval_exp9_covid_nm_to_ukr_rus_lp_to_election20...,/scratch1/singhama/data/experiments/exp9_train...,0.0,2.0,12.0,...,[],[],NaN,election2020,pl,election2020+pl(0),nan|election2020+pl(0),NaN,0.231323,0.211179
71,elfxgenk,eval_exp9_covid_nm_to_ukr_rus_lp_to_election20...,finished,election2020,nm,eval_exp9_covid_nm_to_ukr_rus_lp_to_election20...,/scratch1/singhama/data/experiments/exp9_train...,0.0,3.0,12.0,...,[],[],NaN,election2020,nm,election2020+nm(0),nan|election2020+nm(0),NaN,0.500718,0.327965
72,929y8tq6,eval_exp8_midterm_nm_to_ukr_rus_lp_to_election...,finished,election2020,pl,eval_exp8_midterm_nm_to_ukr_rus_lp_to_election...,/scratch1/singhama/data/experiments/exp8_train...,3.0,2.0,12.0,...,[],[],NaN,election2020,pl,election2020+pl(3),nan|election2020+pl(3),NaN,0.630587,0.573569
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,hk4oxiot,eval_exp1_midterm_nm_to_covid_nm_to_ukr_rus_tw...,finished,ukr_rus_twitter,nm,eval_exp1_midterm_nm_to_covid_nm_to_ukr_rus_tw...,/scratch1/singhama/data/experiments/train2_mid...,0.0,3.0,12.0,...,[],[],NaN,ukr_rus_twitter,nm,ukr_rus_twitter+nm(0),nan|ukr_rus_twitter+nm(0),NaN,0.502104,0.311786
335,1k2p1n96,eval_exp1_midterm_nm_to_covid_nm_to_midterm_nm...,finished,midterm,nm,eval_exp1_midterm_nm_to_covid_nm_to_midterm_nm...,/scratch1/singhama/data/experiments/train2_mid...,0.0,3.0,12.0,...,[],[],NaN,midterm,nm,midterm+nm(0),nan|midterm+nm(0),NaN,0.499859,0.304898
336,z9uzzzzb,eval_exp1_midterm_nm_to_covid_nm_to_election20...,finished,election2020,nm,eval_exp1_midterm_nm_to_covid_nm_to_election20...,/scratch1/singhama/data/experiments/train2_mid...,0.0,3.0,12.0,...,[],[],NaN,election2020,nm,election2020+nm(0),nan|election2020+nm(0),NaN,0.500880,0.317415
337,ilbsjc76,eval_exp1_midterm_nm_to_covid_nm_to_covid19_tw...,finished,covid19_twitter,nm,eval_exp1_midterm_nm_to_covid_nm_to_covid19_tw...,/scratch1/singhama/data/experiments/train2_mid...,0.0,3.0,12.0,...,[],[],NaN,covid19_twitter,nm,covid19_twitter+nm(0),nan|covid19_twitter+nm(0),NaN,0.500386,0.313088
